# 04 — Genre Clustering & ENOA Space

Deep dive into the Every Noise at Once coordinate system, genre taxonomy,
spatial clustering, zone exploration, and cross-playlist genre overlap.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from recommend.modules.genre import (
    expand_genre_zone,
    filter_by_enoa_proximity,
    genre_to_enoa,
    load_genre_map_from_db,
)
from recommend.preprocessing import build_feature_matrix
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

conn = get_connection(read_only=False)
init_schema(conn)
corpus = build_feature_matrix(conn)
genre_map = load_genre_map_from_db(conn)

## 1. ENOA Full Genre Map

Scatter of all 6k+ genres from `genre_xy` — `(left, top)`, colored by `color` column.

In [ ]:
# Load full genre_xy table
genre_xy = conn.execute('SELECT first_genre, top, "left", color FROM genre_xy').pl()
print(f"ENOA genres: {len(genre_xy):,}")

fig = px.scatter(
    genre_xy.to_pandas(),
    x="left", y="top",
    hover_name="first_genre",
    color="color",
    title=f"Every Noise at Once — Full Genre Map ({len(genre_xy):,} genres)",
    width=1000, height=800,
    opacity=0.6,
)
fig.update_yaxes(autorange="reversed")  # ENOA convention: top=0 at top
fig.update_layout(showlegend=False)
fig.show()

## 2. Library Overlay

Overlay library tracks' `(left, top)` onto the genre map.

In [ ]:
# Filter to tracks with real ENOA coords
library_enoa = corpus.filter(
    (pl.col("top") != 0) | (pl.col("left") != 0)
).select(["id", "track_name", "top", "left", "gen_4", "first_genre"])

print(f"Library tracks with ENOA coords: {len(library_enoa):,}")

fig = go.Figure()

# Background: all ENOA genres (light gray)
fig.add_trace(go.Scatter(
    x=genre_xy["left"].to_list(),
    y=genre_xy["top"].to_list(),
    mode="markers",
    marker=dict(size=2, color="lightgray", opacity=0.3),
    text=genre_xy["first_genre"].to_list(),
    hoverinfo="text",
    name="All ENOA genres",
))

# Foreground: library tracks colored by gen_4
gen4_groups = library_enoa.filter(pl.col("gen_4").is_not_null()).partition_by("gen_4", as_dict=True)
colors = {"acoustic": "#2ca02c", "dance": "#ff7f0e", "electronic": "#1f77b4", "instrumental": "#d62728"}

for (gen4_val,), group_df in gen4_groups.items():
    fig.add_trace(go.Scatter(
        x=group_df["left"].to_list(),
        y=group_df["top"].to_list(),
        mode="markers",
        marker=dict(size=4, color=colors.get(gen4_val, "purple"), opacity=0.6),
        text=group_df["track_name"].to_list(),
        hoverinfo="text",
        name=gen4_val,
    ))

fig.update_yaxes(autorange="reversed")
fig.update_layout(
    title="Library Tracks Overlaid on ENOA Genre Map",
    width=1000, height=800,
    xaxis_title="left", yaxis_title="top",
)
fig.show()

## 3. Genre Taxonomy Tree

Visualize the `gen_4 → gen_6 → gen_8 → my_genre` hierarchy.

In [ ]:
# Genre hierarchy from genre_map table
genre_hierarchy = conn.execute("""
    SELECT gen_4, gen_6, gen_8, my_genre, sub_genre, COUNT(*) as n_genres
    FROM genre_map
    WHERE gen_4 IS NOT NULL
    GROUP BY gen_4, gen_6, gen_8, my_genre, sub_genre
    ORDER BY gen_4, gen_6, gen_8, my_genre
""").pl()

print(f"Genre taxonomy entries: {len(genre_hierarchy)}")
genre_hierarchy.head(20)

In [ ]:
# Sunburst: gen_4 → gen_6 → gen_8
# Count library tracks per taxonomy level
tax_cols = ["gen_4", "gen_6", "gen_8"]
available_tax = [c for c in tax_cols if c in corpus.columns]

if len(available_tax) >= 2:
    tax_counts = (
        corpus.filter(pl.col(available_tax[0]).is_not_null())
        .group_by(available_tax)
        .len()
    ).to_pandas()

    fig = px.sunburst(
        tax_counts,
        path=available_tax,
        values="len",
        title="Genre Taxonomy — Library Track Distribution",
        width=800, height=800,
    )
    fig.show()
else:
    print("Insufficient taxonomy columns for sunburst")

## 4. Spatial Clustering on ENOA Coordinates

KMeans and DBSCAN on `(top, left)` for the 6k genre points.
Compare to the curated `gen_4` labels.

In [ ]:
# KMeans on genre_xy coordinates
genre_coords = genre_xy.select(["top", "left"]).to_numpy().astype(np.float64)

# Try k=4 (matching gen_4 cardinality)
for k in [4, 6, 8]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(genre_coords)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        genre_coords[:, 1], genre_coords[:, 0],  # left, top
        c=labels, s=3, alpha=0.5, cmap="tab10",
    )
    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title(f"KMeans k={k} on ENOA Genre Coordinates")
    plt.colorbar(scatter, ax=ax, label="Cluster")
    plt.tight_layout()
    plt.show()

In [ ]:
# DBSCAN — density-based clustering
# Normalize coordinates first since top and left have different scales
from sklearn.preprocessing import StandardScaler

coords_scaled = StandardScaler().fit_transform(genre_coords)
db = DBSCAN(eps=0.3, min_samples=10)
db_labels = db.fit_predict(coords_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
print(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points")

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    genre_coords[:, 1], genre_coords[:, 0],
    c=db_labels, s=3, alpha=0.5, cmap="tab20",
)
ax.set_xlabel("left")
ax.set_ylabel("top")
ax.invert_yaxis()
ax.set_title(f"DBSCAN on ENOA (eps=0.3) — {n_clusters} clusters")
plt.tight_layout()
plt.show()

### 4b. Multi-Algorithm Comparison with Cluster Metrics

Agglomerative + Spectral clustering alongside KMeans/DBSCAN.
Scored by Silhouette, Calinski-Harabasz, and Davies-Bouldin.

In [ ]:
# Multi-algorithm clustering comparison on ENOA coordinates
from sklearn.cluster import AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.preprocessing import MinMaxScaler

N_CLUSTERS = 6
coords_mm = MinMaxScaler().fit_transform(genre_coords)

CLUSTER_MODELS = {
    "KMeans": KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10),
    "Agglomerative": AgglomerativeClustering(n_clusters=N_CLUSTERS),
    "Spectral": SpectralClustering(n_clusters=N_CLUSTERS, random_state=42, affinity="nearest_neighbors"),
}

results = []
fig, axes = plt.subplots(1, len(CLUSTER_MODELS), figsize=(7 * len(CLUSTER_MODELS), 6))

for ax, (name, model) in zip(axes, CLUSTER_MODELS.items()):
    labels = model.fit_predict(coords_mm)

    sil = silhouette_score(coords_mm, labels)
    ch = calinski_harabasz_score(coords_mm, labels)
    db = davies_bouldin_score(coords_mm, labels)
    results.append({"model": name, "silhouette": round(sil, 3), "calinski_harabasz": round(ch, 1), "davies_bouldin": round(db, 3)})

    ax.scatter(genre_coords[:, 1], genre_coords[:, 0], c=labels, s=3, alpha=0.5, cmap="tab10")
    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title(f"{name} (k={N_CLUSTERS})\nsil={sil:.3f}  DB={db:.3f}")

plt.suptitle(f"Clustering Algorithm Comparison (k={N_CLUSTERS})", fontsize=14)
plt.tight_layout()
plt.show()

metrics_df = pl.DataFrame(results)
print("Cluster quality metrics (higher silhouette/CH better, lower DB better):")
metrics_df

### 4c. Cluster Selection Diagnostics

PCA explained variance (scree plot), KMeans elbow curve (inertia),
and silhouette score sweep across k values.

In [ ]:
# PCA scree plot + elbow method + silhouette k-sweep
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score as _sil_score

K_RANGE = range(2, 13)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- PCA explained variance (scree plot) ---
pca_full = PCA(random_state=42).fit(coords_mm)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
axes[0].plot(range(1, len(cumvar) + 1), cumvar, "o-", linewidth=2)
axes[0].set_xlabel("Number of components")
axes[0].set_ylabel("Cumulative explained variance")
axes[0].set_title("PCA Scree Plot (ENOA coordinates)")
axes[0].axhline(0.95, color="red", linestyle="--", alpha=0.5, label="95% threshold")
axes[0].legend()
axes[0].set_xticks(range(1, len(cumvar) + 1))

# --- Elbow method (KMeans inertia) ---
inertias = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(coords_mm)
    inertias.append(km.inertia_)

axes[1].plot(list(K_RANGE), inertias, "o-", linewidth=2)
axes[1].set_xlabel("k (number of clusters)")
axes[1].set_ylabel("Inertia (within-cluster SS)")
axes[1].set_title("KMeans Elbow Method")
axes[1].set_xticks(list(K_RANGE))

# --- Silhouette score sweep ---
sil_scores = []
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(coords_mm)
    sil_scores.append(_sil_score(coords_mm, labels))

best_k = list(K_RANGE)[int(np.argmax(sil_scores))]
axes[2].plot(list(K_RANGE), sil_scores, "o-", linewidth=2)
axes[2].set_xlabel("k (number of clusters)")
axes[2].set_ylabel("Silhouette score")
axes[2].set_title(f"Silhouette Score vs k (best k={best_k})")
axes[2].axvline(best_k, color="red", linestyle="--", alpha=0.5, label=f"k={best_k}")
axes[2].legend()
axes[2].set_xticks(list(K_RANGE))

plt.suptitle("Cluster Selection Diagnostics", fontsize=14)
plt.tight_layout()
plt.show()

print(f"Best silhouette at k={best_k}: {max(sil_scores):.4f}")

## 5. Genre Zone Exploration

Visualize `expand_genre_zone()` circles for selected genres.

In [ ]:
ZONE_GENRES = ["indie rock", "deep house", "bossa nova", "ambient"]
RADIUS = 1500.0

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, genre_name in enumerate(ZONE_GENRES):
    ax = axes[i]
    center = genre_to_enoa(genre_name, genre_map)

    if center is None:
        ax.set_title(f"{genre_name} — not found in ENOA")
        continue

    # Background: all library tracks
    ax.scatter(
        library_enoa["left"].to_numpy(),
        library_enoa["top"].to_numpy(),
        s=1, alpha=0.1, color="gray",
    )

    # Zone circle
    circle = plt.Circle(
        (center[1], center[0]),  # (left, top)
        RADIUS, fill=False, color="red", linewidth=2, linestyle="--",
    )
    ax.add_patch(circle)

    # Tracks in zone
    zone_tracks = filter_by_enoa_proximity(corpus, center, radius=RADIUS)
    if not zone_tracks.is_empty():
        ax.scatter(
            zone_tracks["left"].to_numpy(),
            zone_tracks["top"].to_numpy(),
            s=8, alpha=0.6, color="red",
            label=f"{len(zone_tracks)} tracks in zone",
        )

    # Center point
    ax.scatter([center[1]], [center[0]], s=100, marker="*", color="gold", zorder=5)

    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title(f"{genre_name} zone (r={RADIUS})")
    ax.legend(fontsize=8)
    ax.set_aspect("equal")

plt.suptitle("Genre Zone Exploration", fontsize=14)
plt.tight_layout()
plt.show()

### 5b. Audio Feature Violins by gen_4

Full distribution shape per genre group — richer than boxplots or medians.

In [ ]:
# Violin plots: audio feature distributions by gen_4
VIZ_FEATURES = ["danceability", "energy", "valence", "acousticness", "instrumentalness"]

if "gen_4" in corpus.columns:
    gen4_clean = corpus.filter(pl.col("gen_4").is_not_null())
    gen4_pd = gen4_clean.select(VIZ_FEATURES + ["gen_4"]).to_pandas()

    fig, axes = plt.subplots(1, len(VIZ_FEATURES), figsize=(16, 5))
    for ax, feat in zip(axes, VIZ_FEATURES):
        order = gen4_pd.groupby("gen_4")[feat].median().sort_values().index
        sns.violinplot(data=gen4_pd, x="gen_4", y=feat, order=order, ax=ax, inner="box", cut=0)
        ax.set_xlabel("")
        ax.set_title(feat)
        ax.tick_params(axis="x", rotation=30)
    plt.suptitle("Audio Feature Profiles by gen_4", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No gen_4 column")

### 5c. Radar Charts per gen_4

Median audio feature profile per genre group — spider/radar layout.

In [ ]:
# Radar/spider charts: median audio features per gen_4
RADAR_FEATURES = [
    "danceability", "energy", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
]

if "gen_4" in corpus.columns:
    gen4_clean = corpus.filter(pl.col("gen_4").is_not_null())
    gen4_medians = (
        gen4_clean.group_by("gen_4")
        .agg([pl.col(f).median() for f in RADAR_FEATURES])
        .sort("gen_4")
    )

    n_genres = len(gen4_medians)
    angles = np.linspace(0, 2 * np.pi, len(RADAR_FEATURES), endpoint=False).tolist()
    angles += angles[:1]

    fig, axes = plt.subplots(
        1, n_genres, figsize=(4.5 * n_genres, 4.5), subplot_kw=dict(polar=True),
    )
    if n_genres == 1:
        axes = [axes]

    for ax, row in zip(axes, gen4_medians.iter_rows(named=True)):
        vals = [float(row[f]) for f in RADAR_FEATURES] + [float(row[RADAR_FEATURES[0]])]
        ax.plot(angles, vals, linewidth=2)
        ax.fill(angles, vals, alpha=0.25)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(RADAR_FEATURES, size=8)
        ax.set_ylim(0, 1)
        ax.set_title(row["gen_4"], pad=15)

    fig.suptitle("Median Audio Feature Radar by gen_4", y=1.05, fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("No gen_4 column")

## 6. Genre Coverage Gaps

Which gen_4/gen_6 regions are under-represented?

In [ ]:
# Track count per gen_4
if "gen_4" in corpus.columns:
    gen4_counts = corpus.group_by("gen_4").len().sort("len", descending=True)
    total = len(corpus)
    gen4_counts = gen4_counts.with_columns(
        (pl.col("len") / total * 100).round(1).alias("pct")
    )
    print("gen_4 coverage:")
    print(gen4_counts)

# gen_6 coverage
if "gen_6" in corpus.columns:
    gen6_counts = corpus.group_by("gen_6").len().sort("len", descending=True)
    gen6_counts = gen6_counts.with_columns(
        (pl.col("len") / total * 100).round(1).alias("pct")
    )
    print(f"\ngen_6 coverage ({len(gen6_counts)} groups):")
    gen6_counts

In [ ]:
# ENOA spatial coverage: divide coordinate space into grid, count tracks per cell
if "top" in corpus.columns and "left" in corpus.columns:
    grid_size = 500  # pixels per cell
    tracks_with_coords = corpus.filter((pl.col("top") != 0) | (pl.col("left") != 0))

    top_vals = tracks_with_coords["top"].to_numpy()
    left_vals = tracks_with_coords["left"].to_numpy()

    fig, ax = plt.subplots(figsize=(10, 8))
    h = ax.hist2d(left_vals, top_vals, bins=20, cmap="YlOrRd")
    plt.colorbar(h[3], ax=ax, label="Track count")
    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title("ENOA Spatial Density — Library Tracks")
    plt.tight_layout()
    plt.show()

### 6b. gen_8 Composition per Playlist

Finer-grained view than the gen_4 heatmap — shows gen_8 breakdown per playlist.

In [ ]:
# gen_8 stacked bar chart per playlist (row-normalized)
if "gen_8" in corpus.columns:
    playlist_gen8 = conn.execute("""
        SELECT p.playlist_name, tg.gen_8, COUNT(*) AS n
        FROM playlist_tracks pt
        JOIN playlists p ON pt.playlist_id = p.playlist_id
        LEFT JOIN track_genre tg ON pt.track_id = tg.track_id
        WHERE tg.gen_8 IS NOT NULL
        GROUP BY p.playlist_name, tg.gen_8
    """).pl()

    if not playlist_gen8.is_empty():
        pivot = playlist_gen8.pivot(on="gen_8", index="playlist_name", values="n").fill_null(0)
        gen8_cols = sorted([c for c in pivot.columns if c != "playlist_name"])

        # Row-normalize
        data = pivot.select(gen8_cols).to_numpy().astype(float)
        row_sums = data.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        data_pct = data / row_sums

        playlist_names = pivot["playlist_name"].to_list()

        fig, ax = plt.subplots(figsize=(14, max(6, len(playlist_names) * 0.5)))
        bottom = np.zeros(len(playlist_names))
        cmap = plt.cm.get_cmap("tab20", len(gen8_cols))

        for i, col in enumerate(gen8_cols):
            widths = data_pct[:, i]
            ax.barh(playlist_names, widths, left=bottom, label=col, color=cmap(i))
            bottom += widths

        ax.set_xlabel("Proportion")
        ax.set_title("gen_8 Composition per Playlist (row-normalized)")
        ax.legend(
            bbox_to_anchor=(1.02, 1), loc="upper left",
            fontsize=7, ncol=1,
        )
        plt.tight_layout()
        plt.show()
else:
    print("No gen_8 column in corpus")

## 7. ENOA as Predictor of gen_4

How well do `(top, left)` coordinates predict the `gen_4` label?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

if "gen_4" in corpus.columns:
    # Filter to tracks with both ENOA coords and gen_4 label
    labeled = corpus.filter(
        pl.col("gen_4").is_not_null() &
        ((pl.col("top") != 0) | (pl.col("left") != 0))
    )
    print(f"Labeled tracks with ENOA coords: {len(labeled):,}")

    X = labeled.select(["top", "left"]).to_numpy()
    y = labeled["gen_4"].to_list()

    # Quick KNN cross-validation
    knn = KNeighborsClassifier(n_neighbors=15)
    scores = cross_val_score(knn, X, y, cv=5, scoring="accuracy")
    print(f"KNN (k=15) accuracy from (top, left) → gen_4: {scores.mean():.3f} ± {scores.std():.3f}")

    # Decision boundary visualization
    knn.fit(X, y)
    x_min, x_max = X[:, 1].min() - 100, X[:, 1].max() + 100
    y_min, y_max = X[:, 0].min() - 100, X[:, 0].max() + 100
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200),
    )
    # Note: model expects (top, left) but meshgrid gives (left, top)
    grid_points = np.c_[yy.ravel(), xx.ravel()]
    Z = knn.predict(grid_points)

    # Encode labels to ints for contour
    unique_labels = sorted(set(y))
    label_to_int = {l: i for i, l in enumerate(unique_labels)}
    Z_int = np.array([label_to_int[z] for z in Z]).reshape(xx.shape)
    y_int = np.array([label_to_int[yi] for yi in y])

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.contourf(xx, yy, Z_int, alpha=0.3, cmap="tab10")
    scatter = ax.scatter(X[:, 1], X[:, 0], c=y_int, s=3, alpha=0.5, cmap="tab10")

    handles = [plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=plt.cm.tab10(i / len(unique_labels)),
               markersize=8, label=l) for i, l in enumerate(unique_labels)]
    ax.legend(handles=handles, title="gen_4")
    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title(f"KNN Decision Boundary: (top, left) → gen_4 (acc={scores.mean():.3f})")
    plt.tight_layout()
    plt.show()

## 8. Cross-Playlist Genre Overlap

Jaccard similarity of genre sets between playlists.

In [ ]:
# Build genre sets per playlist
playlist_genres = conn.execute("""
    SELECT p.playlist_name, tg.first_genre
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
    LEFT JOIN track_genre tg ON pt.track_id = tg.track_id
    WHERE tg.first_genre IS NOT NULL
""").pl()

if not playlist_genres.is_empty():
    genre_sets: dict[str, set[str]] = {}
    for name in playlist_genres["playlist_name"].unique().to_list():
        genres = set(
            playlist_genres.filter(pl.col("playlist_name") == name)["first_genre"].to_list()
        )
        genre_sets[name] = genres

    # Jaccard similarity matrix
    names = sorted(genre_sets.keys())
    n = len(names)
    jaccard = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            a, b = genre_sets[names[i]], genre_sets[names[j]]
            if len(a | b) > 0:
                jaccard[i, j] = len(a & b) / len(a | b)

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        jaccard,
        xticklabels=names, yticklabels=names,
        annot=True, fmt=".2f", cmap="YlOrRd",
        square=True, ax=ax,
    )
    ax.set_title("Cross-Playlist Genre Overlap (Jaccard Similarity on first_genre sets)")
    plt.tight_layout()
    plt.show()
else:
    print("No playlist genre data available")

## 9. Genre Map Health

How many library tracks have unmapped `first_genre` values (no gen_4/gen_8)?
Which raw genres are most common among unmapped tracks?

In [ ]:
# Tracks with first_genre but no gen_4 mapping
if "first_genre" in corpus.columns and "gen_4" in corpus.columns:
    unmapped = corpus.filter(
        pl.col("first_genre").is_not_null() & pl.col("gen_4").is_null()
    )
    mapped = corpus.filter(pl.col("gen_4").is_not_null())
    no_genre = corpus.filter(pl.col("first_genre").is_null())

    print(f"Mapped to gen_4:      {len(mapped):,} ({len(mapped)/len(corpus)*100:.1f}%)")
    print(f"Unmapped (has genre): {len(unmapped):,} ({len(unmapped)/len(corpus)*100:.1f}%)")
    print(f"No first_genre at all: {len(no_genre):,} ({len(no_genre)/len(corpus)*100:.1f}%)")

    # Top unmapped genres by track count
    if not unmapped.is_empty():
        top_unmapped = (
            unmapped.group_by("first_genre")
            .len()
            .sort("len", descending=True)
            .head(25)
        )
        print(f"\nTop 25 unmapped genres ({len(top_unmapped)} shown):")

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(
            top_unmapped["first_genre"].to_list()[::-1],
            top_unmapped["len"].to_list()[::-1],
        )
        ax.set_xlabel("Track count")
        ax.set_title("Top Unmapped Genres (have first_genre but no gen_4)")
        plt.tight_layout()
        plt.show()
    else:
        print("All genres with first_genre are mapped — no gaps.")

In [ ]:
# ENOA centroid distance heatmap between top gen_8 groups
if "gen_8" in corpus.columns and "top" in corpus.columns and "left" in corpus.columns:
    # Compute ENOA centroid per gen_8
    gen8_centroids = (
        corpus.filter(
            pl.col("gen_8").is_not_null()
            & ((pl.col("top") != 0) | (pl.col("left") != 0))
        )
        .group_by("gen_8")
        .agg(
            pl.col("top").mean().alias("top_mean"),
            pl.col("left").mean().alias("left_mean"),
            pl.len().alias("n_tracks"),
        )
        .sort("n_tracks", descending=True)
    )

    # Take top groups with enough tracks
    top_groups = gen8_centroids.filter(pl.col("n_tracks") >= 5).head(15)
    group_names = top_groups["gen_8"].to_list()
    centers = top_groups.select(["top_mean", "left_mean"]).to_numpy()

    # Pairwise Euclidean distance
    from scipy.spatial.distance import pdist, squareform

    dist_matrix = squareform(pdist(centers, metric="euclidean"))

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        dist_matrix,
        xticklabels=group_names, yticklabels=group_names,
        annot=True, fmt=".0f", cmap="YlOrRd_r",
        square=True, ax=ax,
    )
    ax.set_title("ENOA Centroid Distance between gen_8 Groups")
    plt.tight_layout()
    plt.show()

## 10. Genre Dendrogram

Hierarchical clustering on scaled audio-feature centroids per gen_4 group (ward linkage).

In [ ]:
# Dendrogram on genre centroids (ward linkage on scaled audio features)
from scipy.cluster.hierarchy import dendrogram, linkage

AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo",
]

# Build centroids per gen_4 (or gen_8 for finer granularity)
for level in ["gen_8", "gen_4"]:
    if level not in corpus.columns:
        continue
    available_af = [f for f in AUDIO_FEATURES if f in corpus.columns]
    centroid_data = (
        corpus.filter(pl.col(level).is_not_null())
        .group_by(level)
        .agg([pl.col(f).mean() for f in available_af])
        .sort(level)
    )
    if len(centroid_data) < 3:
        continue

    labels = centroid_data[level].to_list()
    X_centroids = centroid_data.select(available_af).to_numpy()

    # Scale before linkage
    from sklearn.preprocessing import MinMaxScaler as _MMS
    X_scaled = _MMS().fit_transform(X_centroids)

    Z = linkage(X_scaled, method="ward")
    fig, ax = plt.subplots(figsize=(12, max(5, len(labels) * 0.3)))
    dendrogram(Z, labels=labels, ax=ax, orientation="left", leaf_font_size=9)
    ax.set_title(f"Genre Dendrogram — {level} (ward linkage on scaled audio centroids)")
    ax.set_xlabel("Ward distance")
    plt.tight_layout()
    plt.show()

In [ ]:
conn.close()